# 3 · Class-based nodes and FlowStore

Two advanced features:

1. **`BaseNode` ABC** — for nodes that need state or direct access to the execution request (not just inputs).
2. **`FlowStore`** — a side-channel key/value cache that any node can read/write, outside the edge graph.

In [1]:
from typing import Annotated, Any

from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.execution.engine import collect, execute
from conductor.execution.request import NodeExecRequest
from conductor.execution.store import FlowStore
from conductor.node import BaseNode
from conductor.widgets import Output, Text

registry = NodeRegistry()

## A class-based node

`BaseNode.execute(req)` receives a `NodeExecRequest` with inputs, data, and the full run state. Use it when you need internal state or direct store access.

In [2]:
class WordCounter(BaseNode):
    """Counts words in input text and stashes the count in FlowStore."""

    node_id = "word-counter"
    node_name = "Word Counter"
    node_description = "Counts words and caches the count"

    def execute(self, req: NodeExecRequest) -> Any:
        text = req.inputs.get("text", "")
        count = len(text.split())
        req.state.store.set(f"word_count:{req.node_id}", count)
        return count


registry.register_class(WordCounter)

## FlowStore via type injection

Any function-based node with a `store: FlowStore` parameter gets the store auto-injected. It is NOT treated as a node input — it doesn't appear in the UI or the validation model.

In [3]:
@registry.node("echo", version=1, name="Echo", description="Echoes input")
def echo(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return text


@registry.node(
    "summary-report",
    version=1,
    name="Summary Report",
    description="Generates a report using data from FlowStore",
)
def summary_report(
    text: Annotated[str, Text(label="Text")],
    store: FlowStore,  # auto-injected — not a node input
) -> Annotated[str, Output(label="Report")]:
    counts = {k: store.get(k) for k in store.keys() if k.startswith("word_count:")}
    lines = [f"Text: {text[:50]}..."]
    for key, n in counts.items():
        node_id = key.split(":", 1)[1]
        lines.append(f"  Node {node_id} counted {n} words")
    return "\n".join(lines)

## Build and run

```text
  echo ──> word-counter
      └──> summary-report  (reads word_count:* from the store)
```

In [4]:
long_text = "The quick brown fox jumps over the lazy dog and then does it again"

nodes = [
    GraphNode("n1", "echo@1", {"text": long_text}),
    GraphNode("n2", "word-counter@1", None),
    GraphNode("n3", "summary-report@1", None),
]
edges = [
    GraphEdge("e1", "n1", "n2", "result", "text"),
    GraphEdge("e2", "n1", "n3", "result", "text"),
]

compiled = compile(nodes=nodes, edges=edges, registry=registry)
results = await collect(execute(compiled))

for node_id, result in results.items():
    print(f"{node_id}: {result}")

n1: {'result': 'The quick brown fox jumps over the lazy dog and then does it again'}
n2: {'result': 14}
n3: {'result': 'Text: The quick brown fox jumps over the lazy dog and th...\n  Node n2 counted 14 words'}


## Confirm the store parameter is not a user-facing input

In [5]:
report = registry.get("summary-report@1")
input_names = [inp.name for inp in report.inputs]
print(f"summary-report inputs: {input_names}")
assert "store" not in input_names
print("(FlowStore correctly excluded)")

summary-report inputs: ['text']
(FlowStore correctly excluded)
